# Logistic Regression: Binary Classification

## Simple self-study notes with a pass/fail example

Logistic Regression predicts the **probability** that an example belongs to a class. Despite its name, it is mainly used for **classification**, not for predicting a continuous number.

![Logistic sigmoid curve](https://biometrics-iita.github.io/Logistic-Regression/images/logcurve.png)

Image source: [Logistic Regression Curve](https://biometrics-iita.github.io/Logistic-Regression/). Implementation reference: [scikit-learn LogisticRegression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

## 1. What is binary classification?

Binary classification means there are exactly two output categories.

| Input example | Output classes |
|---|---|
| Study hours | Fail = 0, Pass = 1 |
| Email text | Not spam = 0, Spam = 1 |
| Medical measurements | No disease = 0, Disease = 1 |

In this lesson, the model receives study hours and predicts the probability of **Pass**.

In [ ]:
# Cell 1: create a small pass/fail dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression

study_hours = np.array([1, 2, 3, 4, 4.5, 5, 6, 7, 8, 9]).reshape(-1, 1)
passed = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
data = pd.DataFrame({'Study hours': study_hours.ravel(), 'Pass (1) / Fail (0)': passed})
data

### What did we create?

Study hours is the input feature X. Pass or fail is the target y. The target is categorical, even though we represent the two categories using 0 and 1.

In [ ]:
# Cell 2: draw the binary data
plt.figure(figsize=(8, 4))
plt.scatter(study_hours, passed, s=80, color=['#E45756' if label == 0 else '#54A24B' for label in passed])
plt.yticks([0, 1], ['Fail (0)', 'Pass (1)'])
plt.xlabel('Study hours')
plt.title('A binary classification problem')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 2. Why not use Linear Regression for classification?

A Linear Regression line can predict less than 0 or more than 1. Those values are not valid probabilities. It can also move a lot when an unusual point changes the best-fit line.

A threshold such as 0.5 may turn a linear output into a class label, but it does not fix these problems.

In [ ]:
# Cell 3: compare a linear line with a logistic probability curve
linear_model = LinearRegression().fit(study_hours, passed)
logistic_model = LogisticRegression().fit(study_hours, passed)
hour_grid = np.linspace(0, 12, 300).reshape(-1, 1)
linear_output = linear_model.predict(hour_grid)
pass_probability = logistic_model.predict_proba(hour_grid)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(study_hours, passed, color='#4C78A8', label='Training points')
axes[0].plot(hour_grid, linear_output, color='#E45756', label='Linear Regression')
axes[0].axhline(0.5, linestyle='--', color='gray', label='Example threshold')
axes[0].set_ylim(-0.5, 1.5)
axes[0].set_title('Linear output can leave 0 to 1')
axes[0].set_xlabel('Study hours')
axes[0].legend()

axes[1].scatter(study_hours, passed, color='#4C78A8', label='Training points')
axes[1].plot(hour_grid, pass_probability, color='#54A24B', linewidth=3, label='Logistic probability')
axes[1].axhline(0.5, linestyle='--', color='gray', label='Decision threshold')
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title('Logistic Regression stays between 0 and 1')
axes[1].set_xlabel('Study hours')
axes[1].legend()
plt.tight_layout()
plt.show()

### Key observation

The green S-shaped curve is called the **sigmoid** or logistic function. It squashes any input score into a value between 0 and 1.

Probability of Pass = 1 / (1 + exp(-z)), where z is the weighted linear score. A probability is not the same thing as the final class label; the threshold makes that final decision.

In [ ]:
# Cell 4: see how an unusual point can change a linear best-fit line
hours_with_outlier = np.vstack([study_hours, [[12]]])
labels_with_outlier = np.append(passed, 1)
linear_with_outlier = LinearRegression().fit(hours_with_outlier, labels_with_outlier)

plt.figure(figsize=(8, 4))
plt.scatter(hours_with_outlier, labels_with_outlier, color='#4C78A8', label='Data including new point')
plt.plot(hour_grid, linear_model.predict(hour_grid), label='Linear line before', color='#F58518')
plt.plot(hour_grid, linear_with_outlier.predict(hour_grid), label='Linear line after', color='#E45756')
plt.axhline(0.5, color='gray', linestyle='--')
plt.ylim(-0.5, 1.5)
plt.xlabel('Study hours')
plt.title('Linear Regression line can shift when data changes')
plt.legend()
plt.show()

The diagram explains the transcript’s warning: a best-fit line is not naturally designed for a 0/1 target. Logistic Regression directly models class probability instead.

In [ ]:
# Cell 5: make predictions with Logistic Regression
new_hours = np.array([[3], [4.5], [5], [7]])
probabilities = logistic_model.predict_proba(new_hours)[:, 1]
predicted_classes = logistic_model.predict(new_hours)

prediction_table = pd.DataFrame({
    'Study hours': new_hours.ravel(),
    'Probability of pass': probabilities.round(3),
    'Predicted class': predicted_classes,
    'Meaning': np.where(predicted_classes == 1, 'Pass', 'Fail')
})
prediction_table

### Read the prediction code

- fit learns the relationship from labelled examples.
- predict_proba returns probabilities for both classes; column 1 is the probability of Pass.
- predict applies the model’s decision threshold and returns 0 or 1.
- A 0.5 threshold is common, but it can be changed when false positives and false negatives have different costs.

In [ ]:
# Cell 6: visualise how a threshold turns probability into a class
thresholds = [0.3, 0.5, 0.7]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hour_grid, pass_probability, color='#54A24B', linewidth=3, label='Probability of pass')
for threshold in thresholds:
    ax.axhline(threshold, linestyle='--', label=f'Threshold = {threshold}')
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Study hours')
ax.set_ylabel('Predicted probability')
ax.set_title('The threshold is a decision rule')
ax.legend()
plt.show()

## Quick revision card

1. Logistic Regression is a classification algorithm despite its name.
2. Binary classification has two classes, often encoded as 0 and 1.
3. Linear Regression can give values below 0 or above 1, so it is not a natural probability model.
4. The sigmoid curve converts a score to a probability between 0 and 1.
5. predict_proba gives probability; predict gives a class after using a threshold.
6. A threshold is a business decision, not always automatically 0.5.

**One-line interview answer:** Logistic Regression uses a sigmoid function to model the probability of a class, then uses a threshold to classify the example.